# Comandos para realização do trabalho da matéria de NLP com uso da biblioteca SKlearn e NLTK.

## <font color=red>Observação importante:</font>

<font color=yellow>Trabalho realizado com uso de corpus diferente do Fake.br não será aceito!</font>

## Carregando arquivos `pre-processed.csv`, de imagem e de funções auxiliares para dentro do Google Colab

In [ ]:
!wget https://raw.githubusercontent.com/roneysco/Fake.br-Corpus/master/preprocessed/pre-processed.csv -O pre-processed.csv
!git clone https://github.com/N-CPUninter/NLP.git
!rm ./NLP/*.ipynb
!mv ./NLP/* .
!rm -r NLP

## Instalação manual das dependências para uso do SKlearn e do NLTK no Google Colab

In [ ]:
import pandas as pd
import nltk
from nltk import ngrams
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from funcoes_auxiliares import gerar_nuvem_palavras

nltk.download("all")

## Criar dataframe do CSV utilizando o método read_csv do pandas

In [ ]:
df = pd.read_csv('pre-processed.csv')

# PRÁTICA 1 - CRIAÇÃO DE MODELO DE CLASSIFICAÇÃO SUPERVISIONADO PARA ANÁLISE DE FAKE NEWS.

1. Realize os seguintes procedimentos de limpeza dos textos do dataframe criado:

  1.1. Tokenizar

  1.2. Retirar os acentos e números

  1.3. Deixar tudo em minúsculas

  1.4. Retirar as stopwords e pontuações

  1.5. Deixar palavras apenas com radical (STEM)

  1.6. Realizar truncamento dos pares de notícias verdadeiras com falsas para normalizar quantidade de palavras

  1.7. Remontar as notícias em string e criar coluna no dataframe para o resultado deste pré-processamento.

2. Criar matriz de frequências TF-IDF com ngramas de 1 a 3 palavras.

3. Usar a função train_test_split do Scikit Learn para dividir o corpus pré-tratado em 75% dos textos para treinamento e 25% para testes de precisão (usar random_state = 42 ou outro número de sua escolha).

4. Fazer regressão logística com solver = 'lbfgs'.

5. Realizar predição dos textos de teste com o método predict_proba, que retornará a porcentagem predita para fake e para real em um array.

6. Por fim, com as porcentagens calculadas para cada texto de teste, usar a função accuracy_score da biblioteca Scikit Learn para calcular a acurácia geral do algoritmo.

In [ ]:
import re
import nltk
import unicodedata
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# baixando recursos necessários
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('rslp') # Added: Download the 'rslp' stemmer resource

# stopwords em português e stemmer
stop_words = set(stopwords.words('portuguese'))
stop_words.add('nao')
stemmer = RSLPStemmer()

# função do pré-processamento do texto
def preprocess_text(text):

    # retirando os acentos pra evitar palavras duplicadas
    text = unicodedata.normalize('NFD', text)
    text = text.encode('ascii', 'ignore').decode('utf-8')

    # deixando tudo em minúsculo
    text = text.lower()

    # remoção de números
    text = re.sub(r'\d+', '', text)

    # remoção de pontuação
    text = re.sub(r'[^\w\s]', '', text)

    # separação do texto em palavras (tokenização)
    tokens = word_tokenize(text)

    # remoção das palavras muito comuns que não agregam valor
    tokens = [t for t in tokens if t not in stop_words]

    # redução das palavras ao radical (stemming)
    tokens = [stemmer.stem(t) for t in tokens]

    return tokens

# função simples pra limitar o tamanho dos textos
def truncate_tokens(tokens, max_len=100):
    return tokens[:max_len]

# --- DATAFRAME ---

# aplicação do pré-processamento
df['tokens'] = df['preprocessed_news'].apply(preprocess_text)

# truncamento pra padronizar tamanho
df['tokens_trunc'] = df['tokens'].apply(lambda x: truncate_tokens(x, 100))

# junção de volta em uma string (porque o TF-IDF precisa disso)
df['text_processed'] = df['tokens_trunc'].apply(lambda x: ' '.join(x))

# transformação do texto em números usando TF-IDF com ngramas de 1 a 3
vectorizer = TfidfVectorizer(ngram_range=(1,3))
X = vectorizer.fit_transform(df['text_processed'])

# variável alvo (fake ou real)
y = df['label'].map({'fake': 1, 'true': 0})
print(y.unique())

# separação em treino e teste (75% treino, 25% teste)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# modelo de regressão logística
model = LogisticRegression(solver='lbfgs', max_iter=1000)

# treinamento do modelo
model.fit(X_train, y_train)

# predição em forma de probabilidade
y_proba = model.predict_proba(X_test)

# conversão de probabilidades para classe final (0 ou 1)
y_pred = np.argmax(y_proba, axis=1)

# cálculo da acurácia do modelo
accuracy = accuracy_score(y_test, y_pred)

print("Acurácia:", accuracy)

# relatório
print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Package rslp is already up-to-date!


## QUESTÃO 01: Apresente aqui o código referente ao modelo gerado e a nuvem de palavras que foram usadas para identificar textos VERDADEIROS.

1. Formate seus dados usados para o treinamento do seu modelo em um dicionário de tokens e suas frequências.
2. Separe os dados em um grupo com textos marcados como verdadeiros e outro com os falsos.
3. Use a função gerar_nuvem_palavras(dic_de_frequências_textos_verdadeiras, imagem de sua escolha) para gerar a nuvem de palavras


In [ ]:
import re
import nltk
import unicodedata
import numpy as np
import pandas as pd
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import RSLPStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from PIL import Image

nltk.download('punkt')
nltk.download('stopwords')

# stopwords e stemmer
stop_words = set(stopwords.words('portuguese'))
stop_words.add('nao')
stemmer = RSLPStemmer()

# função de pré-processamento
def preprocess_text(text):
    text = unicodedata.normalize('NFD', text)
    text = text.encode('ascii', 'ignore').decode('utf-8')
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

# truncamento
def truncate_tokens(tokens, max_len=100):
    return tokens[:max_len]

# --- PRÉ-PROCESSAMENTO ---
df['tokens'] = df['preprocessed_news'].apply(preprocess_text)
df['tokens_trunc'] = df['tokens'].apply(lambda x: truncate_tokens(x, 100))
df['text_processed'] = df['tokens_trunc'].apply(lambda x: ' '.join(x))

# --- LABEL NUMÉRICO ---
df['label'] = df['label'].map({'fake': 1, 'true': 0})

# Remover linhas onde 'label' é NaN para evitar o erro de treinamento do modelo
df.dropna(subset=['label'], inplace=True)

# Converter a coluna 'label' para tipo inteiro após remover NaNs
df['label'] = df['label'].astype(int)

# --- MODELO ---
vectorizer = TfidfVectorizer(ngram_range=(1,3))
X = vectorizer.fit_transform(df['text_processed'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = LogisticRegression(solver='lbfgs', max_iter=1000)
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)
y_pred = np.argmax(y_proba, axis=1)

accuracy = accuracy_score(y_test, y_pred)
print("Acurácia:", accuracy)
print(classification_report(y_test, y_pred))

# =======================================
# NUVEM DE PALAVRAS (TEXTOS VERDADEIROS)
# =======================================

# separação dos textos verdadeiros (label = 0)
df_true = df[df['label'] == 0]

# junção dos tokens
tokens_true = []
for tokens in df_true['tokens_trunc']:
    tokens_true.extend(tokens)

# criação de dicionário de frequências
freq_true = dict(Counter(tokens_true))

print("Exemplo do dicionário de frequências:")
print(dict(list(freq_true.items())[:10]))

# função fornecida
def gerar_nuvem_palavras(freq_dict, imagem=None):

    if imagem is not None:
        mask = np.array(Image.open(imagem))
    else:
        mask = None

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        mask=mask
    ).generate_from_frequencies(freq_dict)

    plt.figure(figsize=(10,5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.show()

# Variável exigida com o número do RU (mesmo sem uso)
contador4960399 = 0

# Forçando o RU no dicionário com a maior frequência para garantir destaque na nuvem
if freq_true:
    freq_true['4960399'] = max(freq_true.values()) + 1000
else:
    freq_true['4960399'] = 1000

# gerar nuvem de palavras
gerar_nuvem_palavras(freq_true, imagem=r"C:\Users\Vinicius\Downloads\joinha.png")


## QUESTÃO 02: Apresente aqui o código referente ao modelo gerado e a nuvem de palavras que foram usadas para identificar textos FALSOS.

1. Use a função gerar_nuvem_palavras(dic_de_frequências_textos_verdadeiras, imagem de sua escolha) para gerar a nuvem de palavras

In [ ]:
# Coloque seu código aqui


---

# Material Complementar

## Alguns exemplos de uso da função auxiliar
`gerar_nuvem_palavras(dicionario_tokens_e_frequencia, arquivo_mascara)`

Gera uma nuvem de palavras com base em seu dicionário de palavras ou ngramas
    como a chave e a frequência de aparição do token como valor (inteiro).

    Parâmetros:
        dicionario_tokens_e_frequencia (dict): O dicionário de tokens e suas
                                               respectivas frequências de
                                               aparição nos textos.
        arquivo_mascara (str): O nome do arquivo da imagem de máscara. Pde ser:
                                            cloud_mask.png
                                            mapa_brasil_mask.png
                                            thumbs_up_mask.png        
                                            thumbs_down_mask.png
                                            <Outro arquivo de sua escolha>

    Exemplos de Uso:
        1. Para gerar uma nuvem de palavras na máscara mapa do brasil:
            gerar_nuvem_palavras(dicionario_tokens_e_frequencia=word_dict,
                                 arquivo_mascara='mapa_brasil_mask.png')

In [ ]:
# Exemplo com dicionário de tokens unigramas e multigramas com frequências
words_dict = {'Olá aluno':1,
              'ALTERAR':4,
              'bigramas aqui':2,
              'palavras distintas de exemplo':2,
              'Para criar sua nuvem de palavras':1,
              'você deve gerar um dicionário':2,
              'de frequências de palavras e tokens':1}
gerar_nuvem_palavras(dicionario_tokens_e_frequencia=words_dict,
                     arquivo_mascara="mapa_brasil_mask.png")

In [ ]:
# Exemplo com uma frase completa
texto = """Exemplo: Gerar uma nuvem de palavras usando texto completo você deve
           primeiro separar ele um um ou mais tokens, para só depois vetorizar.
           Ao vetorizar você terá a bag of words, que te permitirá contar
           quantas vezes cada palavra ou grupo de palavras aparecem no texto.
           Por fim, basta criar um dicionário contendo a chave como o seu token
           e o valor como a frequência de aparição deste tokem."""
# Vetorização e contagem de frequência simples de bigramas:
vectorizer = CountVectorizer(ngram_range=(2, 2))
bag_of_words = vectorizer.fit_transform([a+' '+b for a,b in (ngrams(texto.split(),2))])
sum_words = bag_of_words.sum(axis=0)
words_freq = [(word, sum_words[0, idx]) for word, idx in vectorizer.vocabulary_.items()]
words_freq =sorted(words_freq, key = lambda x: x[1], reverse=True)
words_dict = dict(words_freq)
print(words_dict)

gerar_nuvem_palavras(dicionario_tokens_e_frequencia=words_dict,
                      arquivo_mascara='thumbs_up_mask.png')